In [1]:
import sys 
from pathlib import Path 

import pandas as pd
import numpy as np

import calendar
from datetime import datetime
from datetime import date

import plotly.graph_objects as go


# Resolve path to project root
root_path = Path.cwd().parents[1]
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

from src.queries.smhi import *


## Data query imports

- [ ] adjust how long format to wide format is transformed

In [ ]:
params_meteo={
    'version' : '1.0',
    'parameter' : ['1', '4', '6', '7', '39'],    # See here for more details https://opendata.smhi.se/metobs/resources/parameter
    'station' : ['63510', '63590'],    # This is the ID of the meteorolical station, which can be found here https://www.smhi.se/data/hitta-data-for-en-plats/ladda-ner-vaderobservationer/precipitationHourlySum
    'period' : 'corrected-archive',
    'data' : 'csv'
}

params_meteo_latest={
    'version' : '1.0',
    'parameter' : ['1', '4', '6', '7', '39'],    # See here for more details https://opendata.smhi.se/metobs/resources/parameter
    'station' : ['63510', '63590'],    # This is the ID of the meteorolical station, which can be found here https://www.smhi.se/data/hitta-data-for-en-plats/ladda-ner-vaderobservationer/precipitationHourlySum
    'period' : 'latest-months',
    'data'  : 'json'
}

df_metops = SMHI_Metops(params_meteo)
df_metops_latest = SMHI_Metops(params_meteo_latest)
df_metops = pd.merge(df_metops, df_metops_latest, how='outer')

In [4]:
params_hydro={
    'version' : 'latest',
    'parameter' : ['1', '3'],    # See here for more details https://opendata.smhi.se/hydroobs/resources/parameter
    'station' : ['1952'],     # This is the ID of the hydrological station, which can be found here https://www.smhi.se/data/hitta-data-for-en-plats/ladda-ner-observationer-fran-sjoar-och-vattendrag/waterLevel
    'period' : 'corrected-archive',
}

df_hydrops = SMHI_Hydroops(params_hydro)

# Transform df_hydrops from long format into wide format
df_hydrops_wide = df_hydrops.pivot_table(
    index='DateTime', 
    columns='StationName', 
    values=['Vattenföring (Dygn) [m³/s]'],
    aggfunc='mean'
)

# Flatten the multi-level df hydrops
df_hydrops_wide.columns = [f"{metric}_{station}" for metric, station in df_hydrops_wide.columns]
df_hydrops_wide = df_hydrops_wide.reset_index()

Request SMHI Open Data Hydrological Observations.
